# 04a · 기본 재생 정리와 재생-보상 (Elementary Renewal Theorem and Renewal-Reward Processes)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §3.1 (N(t)/t → 1/μ, 재생-보상 정리와 예제) · Ross 12e §7.1–7.4 (7.2 N(t)의 분포, 7.3 극한 정리와 Wald 항등식, 7.4 재생-보상, 수명 교체 예제) · G&S 4e §10.1–10.2, §10.5 (엄밀한 증명) · Lawler 2e Ch. 6 (재생 방정식) |
| 선수 노트북 | 00b, 00c, 02a, 02d, 03d, 03e |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **재생(renewal)은 기억이 리셋되는 시각이고, 리셋 사이의 주기(cycle)가 i.i.d.이므로 모든 장기 비율은 "한 주기의 기댓값 두 개의 비"로 환원된다.**

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy import stats, special, optimize
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.renewal import renewal_times, renewal_count, renewal_reward
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("04a")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03e) M/M/c 대기행렬에서 Erlang C 공식 $C(c,a)$, $a=\lambda/\mu$는 무엇의 확률인가? 어떤 정리 덕분에 '시간 평균'이 '도착 고객이 보는 확률'과 같아지는가?

<details><summary>정답</summary>

모든 서버가 바쁜 상태의 정상확률 $\sum_{n\ge c}\pi_n$, 즉 도착 고객이 대기해야 할 확률 $P(W_q>0)$이다. PASTA(Poisson Arrivals See Time Averages) 덕분에 정상분포(시간 평균)가 곧 푸아송 도착 고객이 보는 분포이다.

</details>

**Q2.** (03e) M/M/c/K에서 $K=c$(대기 공간 없음)일 때 손실 확률 공식의 이름과 형태는? 이 공식이 서비스 시간 분포에 대해 갖는 특별한 성질은?

<details><summary>정답</summary>

Erlang B: $B(c,a)=\dfrac{a^c/c!}{\sum_{k=0}^{c}a^k/k!}$. 서비스 분포의 평균만 관여하고 모양에는 무관하다(insensitivity).

</details>

**Q3.** (03e) 같은 도착률 $\lambda$에서 속도 $2\mu$ 서버 1대(M/M/1)와 속도 $\mu$ 서버 2대(M/M/2) 중 평균 체류시간 $W$가 짧은 쪽은? 대기열 길이 $L_q$는?

<details><summary>정답</summary>

$\rho=\lambda/(2\mu)$라 두면 $W_1=\frac{1}{2\mu(1-\rho)}$, $W_2=\frac{1}{\mu(1-\rho^2)}$이고 $W_2/W_1=\frac{2}{1+\rho}>1$: 빠른 서버 1대가 항상 유리하다. 반면 $L_{q,2}/L_{q,1}=\frac{2\rho}{1+\rho}<1$: 대기열만 보면 2대가 짧다. 서비스 중인 시간까지 포함하면 한 대가 이긴다.

</details>

**Q4 (연결 고리).** (03d) 리틀의 법칙 $L=\lambda W$가 성립하려면 도착이 푸아송이거나 서비스가 지수분포여야 하는가? M/M/1에서 $L$은?

<details><summary>정답</summary>

아니다. 안정적인 어떤 대기 시스템에서도 표본경로 수준에서 성립한다(분포 가정 없음). M/M/1: $L=\rho/(1-\rho)$, $\rho=\lambda/\mu$. — 리틀의 법칙이 분포에 무관한 이유가 바로 오늘 배울 재생-보상 논법이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **재생 과정(renewal process).** 서로 독립이고 같은 분포 $F$를 따르는 양의 확률변수 $X_1,X_2,\dots$ (도착 간격, $P(X>0)=1$, $\mu=E[X]\in(0,\infty]$)에 대해
  $$S_0=0,\qquad S_n=X_1+\cdots+X_n\ (\text{재생 시각}),\qquad N(t)=\max\{n: S_n\le t\}\ (\text{시각 } t \text{까지의 재생 횟수}).$$
  푸아송 과정(02a)은 $F$가 지수분포인 특수한 경우다.
- **재생 함수(renewal function).** $m(t)=E[N(t)]=\sum_{n\ge1}P(S_n\le t)$. 모든 $t$에서 유한하다 (Ross 12e §7.2).
- **재생-보상 과정(renewal-reward process).** $i$번째 주기에 보상 $R_i$가 붙고 쌍 $(X_i,R_i)$가 i.i.d.($R_i$는 $X_i$에 의존해도 됨), $E|R|<\infty$. 누적 보상 $R(t)=\sum_{i=1}^{N(t)}R_i$ (주기 끝에 보상을 몰아서 받는다고 약속).
- **수명 교체 정책(age replacement policy).** 부품 수명 $X\sim F$. 고장 나면 즉시 교체(비용 $c_2$), 고장 나지 않아도 나이 $T$가 되면 예방 교체(비용 $c_1<c_2$). 주기 길이는 $\min(X,T)$, 주기 비용은 $c_1\mathbf 1\{X>T\}+c_2\mathbf 1\{X\le T\}$.
- **위험률(hazard rate).** $r(x)=f(x)/\bar F(x)$, $\bar F=1-F$. IFR(increasing failure rate)은 $r$이 증가함수인 것. 지수분포는 $r$ 상수, Weibull(모양 $k>1$)은 IFR. Weibull(모양 2, 척도 1): $\bar F(x)=e^{-x^2}$, $r(x)=2x$, $\mu=\Gamma(3/2)=\sqrt\pi/2\approx0.8862$.
- **격자(lattice, arithmetic) 분포.** $X$가 어떤 $d>0$의 정수배 값만 취하는 경우. 이 노트북의 정리들은 격자 여부와 무관하지만 04b의 극한분포 정리는 비격자를 요구한다.

### 2.2 정리 1 — 재생 과정의 강대수법칙 (Durrett 3e §3.1 Thm 3.1, Ross 12e §7.3 Prop 7.1)

> $\mu<\infty$이면 $\dfrac{N(t)}{t}\to\dfrac1\mu$ a.s. ($t\to\infty$). $\mu=\infty$이면 극한은 0.

**가정이 왜 필요한가**
- i.i.d.: $S_n/n\to\mu$ (강대수법칙)를 쓰기 위해. 독립이 깨져도 에르고딕 정리로 대체 가능하지만 i.i.d.가 가장 깨끗하다.
- $P(X>0)=1$: $N(t)<\infty$ a.s.를 보장. $X=0$에 양의 확률이 있으면 한 시각에 여러 재생이 쌓여 정의가 흔들린다.
- $\mu$가 유한하지 않아도 되는 이유: $X\wedge M$으로 절단하면 $\mu_M\uparrow\infty$이므로 극한이 0임을 보일 수 있다.

<details><summary>증명 (샌드위치 논법, 완전 증명)</summary>

정의에서 $S_{N(t)}\le t<S_{N(t)+1}$ (시각 $t$는 $N(t)$번째 재생 이후, $N(t)+1$번째 재생 이전). 양변을 $N(t)$로 나누면

$$\frac{S_{N(t)}}{N(t)}\;\le\;\frac{t}{N(t)}\;<\;\frac{S_{N(t)+1}}{N(t)+1}\cdot\frac{N(t)+1}{N(t)}.$$

(i) $t\to\infty$일 때 $N(t)\to\infty$ a.s.: 각 $S_n<\infty$ a.s.이므로 $t\ge S_n$이 되는 순간부터 $N(t)\ge n$이다.
(ii) 강대수법칙으로 $S_n/n\to\mu$ a.s. 이고, (i)에 의해 $N(t)$는 $\infty$로 가는 정수열이므로 부분열 논법으로 $S_{N(t)}/N(t)\to\mu$ a.s. 같은 이유로 $S_{N(t)+1}/(N(t)+1)\to\mu$, 그리고 $(N(t)+1)/N(t)\to1$.
(iii) 따라서 좌변과 우변이 모두 $\mu$로 수렴하고 샌드위치로 $t/N(t)\to\mu$, 즉 $N(t)/t\to1/\mu$ a.s. $\square$

$\mu=\infty$인 경우: $X^M_i=\min(X_i,M)$으로 만든 과정은 $N_M(t)\ge N(t)$이므로 $\limsup N(t)/t\le1/\mu_M$이고 $M\to\infty$이면 $\mu_M\uparrow\infty$ (단조수렴). 그래서 극한은 0. $\square$

</details>

### 2.3 정리 2 — 기본 재생 정리 (elementary renewal theorem; Ross 12e §7.3, Durrett 3e §3.1)

> $\dfrac{m(t)}{t}\to\dfrac1\mu$ ($\mu=\infty$이면 0). 즉 **기댓값 수준에서도** 같은 비율.

**가정이 왜 필요한가 / 왜 별도의 정리인가**
- a.s. 수렴에서 기댓값 수렴이 자동으로 따라오지 않는다(유계수렴 또는 균등적분가능성이 필요). 그래서 별도의 증명이 필요하다: 이것이 이 정리가 '따로' 있는 이유다.
- $N(t)+1$이 **정지시간(stopping time)** 이라는 사실이 핵심: $\{N(t)+1=n\}=\{S_{n-1}\le t<S_n\}$은 $X_1,\dots,X_n$으로 결정된다. 반면 $N(t)$는 정지시간이 아니다($X_{N(t)+1}$을 봐야 $N(t)=n$인지 안다).

**증명 스케치.** 정지시간 $\tau=N(t)+1$에 대한 **Wald 항등식** $E[S_\tau]=\mu\,E[\tau]$ (05 모듈에서 증명)을 쓰면
$$E[S_{N(t)+1}]=\mu\,(m(t)+1).$$
- 하한: $S_{N(t)+1}>t$이므로 $\mu(m(t)+1)>t$, 즉 $m(t)/t>1/\mu-1/t$ → $\liminf m(t)/t\ge1/\mu$.
- 상한: $X_i$를 $M$에서 절단한 과정 $N_M(t)\ge N(t)$에 대해 $S^M_{N_M(t)+1}\le t+M$이므로 $\mu_M(m_M(t)+1)\le t+M$, 따라서 $\limsup m(t)/t\le\limsup m_M(t)/t\le1/\mu_M\to1/\mu$ ($M\to\infty$).

상세는 Ross 12e §7.3 또는 Durrett 3e §3.1. 노트북에서는 Erlang(2,1)의 정확한 $m(t)$로 확인한다.

### 2.4 정리 3 — 재생-보상 정리 (renewal reward theorem; Durrett 3e §3.1, Ross 12e §7.4, G&S 4e §10.5)

> $(X_i,R_i)$ i.i.d., $E|R|<\infty$, $\mu<\infty$이면
> $$\frac{R(t)}{t}\to\frac{E[R]}{E[X]}\ \text{a.s.}\qquad\text{그리고}\qquad \frac{E[R(t)]}{t}\to\frac{E[R]}{E[X]}.$$

**가정이 왜 필요한가**
- 쌍 $(X_i,R_i)$가 i.i.d.: 주기 내부에서 $R_i$와 $X_i$의 의존은 허용된다(수명 교체가 정확히 그 경우). 필요한 것은 주기끼리의 독립·동일분포.
- $E|R|<\infty$: $\sum_{i\le n}R_i/n\to E[R]$ (강대수법칙)에 필요.
- $\mu<\infty$: 그래야 $N(t)/t\to1/\mu>0$이고 비율이 의미를 가진다.

<details><summary>증명 (a.s. 부분은 완전 증명, 기댓값 부분은 스케치)</summary>

곱으로 쪼갠다:
$$\frac{R(t)}{t}=\frac{\sum_{i=1}^{N(t)}R_i}{N(t)}\cdot\frac{N(t)}{t}.$$
첫 인자: $N(t)\to\infty$ a.s. (정리 1 증명의 (i))와 강대수법칙 $\sum_{i\le n}R_i/n\to E[R]$에 부분열 논법을 적용하면 $E[R]$로 수렴. 둘째 인자: 정리 1로 $1/\mu$. 곱의 극한은 극한의 곱이므로 $R(t)/t\to E[R]/\mu$ a.s. $\square$

기댓값 버전: $\sum_{i\le N(t)+1}R_i$에 Wald 항등식을 적용하면 $E\big[\sum_{i\le N(t)+1}R_i\big]=E[R]\,(m(t)+1)$이고 기본 재생 정리로 $E[R](m(t)+1)/t\to E[R]/\mu$. 남는 항 $E[R_{N(t)+1}]/t\to0$은 $|R|$을 절단해 보인다(Ross 12e §7.4). $\square$

**결론의 형태를 반드시 기억:** '장기 비율 = 기대 보상 / 기대 주기 길이' $=E[R]/E[X]$이지 $E[R/X]$가 **아니다**.

</details>

### 2.5 정리 4 — 수명 교체 비용률과 최적 조건 (Ross 12e §7.4 예제)

> 장기 평균 비용률
> $$C(T)=\frac{c_1\bar F(T)+c_2F(T)}{\int_0^T\bar F(x)\,dx}.$$
> $F$가 밀도 $f$를 갖고 위험률 $r=f/\bar F$가 **강한 의미로** 증가(strictly IFR)하며 $g(\infty):=\lim_{T\to\infty}g(T)>\dfrac{c_1}{c_2-c_1}$이면, 조건식
> $$g(T):=r(T)\int_0^T\bar F(x)\,dx-F(T)=\frac{c_1}{c_2-c_1}$$
> 은 유일한 해 $T^*$를 갖고 이것이 $C$의 유일한 정류점이자 최소점이며, 최소값은 $C(T^*)=(c_2-c_1)\,r(T^*)$. $C(\infty)=c_2/\mu$ (예방 교체 없음).

**가정이 왜 필요한가**
- $c_1<c_2$: 아니면 예방 교체가 무의미($T^*=\infty$).
- 밀도 $f$: 위험률 $r=f/\bar F$를 정의하고 $C'(T)$를 계산하려면 $F$가 연속인 것만으로는 부족하다.
- 강한 IFR(위험률이 순증가): $g'(T)=r'(T)\int_0^T\bar F>0$이고 $g(0)=0$이므로 $g$는 0에서 출발해 순증가한다 — 해가 있으면 **유일**하다. 약한 IFR(지수분포처럼 $r$ 상수)만 가정하면 $g\equiv0$이 되어 해가 없다(E3에서 깨본다).
- $g(\infty)>c_1/(c_2-c_1)$: 해의 **존재** 조건. $r(T)\to\infty$인 분포(Weibull 모양 $>1$, 균등분포)는 $g(\infty)=\infty$라 자동으로 만족하지만, 위험률이 유한한 값으로 포화하는 분포에서는 $c_1/(c_2-c_1)$이 그보다 크면 예방 교체가 이득이 아니다($T^*=\infty$).
- 재생-보상 정리의 조건: 주기 $\min(X_i,T)$와 비용 쌍이 i.i.d., 기대 주기 $E[\min(X,T)]=\int_0^T\bar F>0$, 비용 유계.

**증명 스케치.** 재생-보상 정리: 분자 $E[\text{cost}]=c_1\bar F(T)+c_2F(T)=c_1+(c_2-c_1)F(T)=:A(T)$, 분모 $E[\min(X,T)]=\int_0^T\bar F(x)dx=:B(T)$ (꼬리 적분 공식). $A'=(c_2-c_1)f(T)$, $B'=\bar F(T)$. $C'=0\iff A'B=AB'\iff(c_2-c_1)f(T)\int_0^T\bar F=(c_1+(c_2-c_1)F(T))\bar F(T)$; 양변을 $(c_2-c_1)\bar F(T)$로 나누면 조건식 $g(T)=c_1/(c_2-c_1)$. 그때 $C=A/B=A'/B'=(c_2-c_1)r(T^*)$.

Weibull(2,1), $c_1=1,c_2=5$: 조건식은 $g(T)=2T\cdot\frac{\sqrt\pi}{2}\mathrm{erf}(T)-(1-e^{-T^2})=\frac14$이고 최소값은 $C(T^*)=(c_2-c_1)\,r(T^*)=8T^*$가 된다. 해 $T^*$의 값은 3절에서 먼저 예측한 뒤 4절에서 코드로 구한다.

### 2.6 직관

재생 과정은 '기억이 리셋되는 시각'의 열이다. 리셋 사이의 구간(주기)이 i.i.d.이므로 긴 시간 뒤의 모든 비율 문제는 '한 주기의 기댓값 두 개의 비'로 환원된다 — 이것이 이 모듈 전체와 04c(마르코프 사슬의 재생 관점), 05(정지시간)로 이어지는 단 하나의 아이디어다.

Erlang(2,1) 예제는 특히 투명하다: 비율 1 푸아송 점열에서 **짝수 번째 점만 남기면** Erlang(2) 재생 과정이므로 $N(t)=\lfloor M(t)/2\rfloor$, $M(t)\sim\text{Poisson}(t)$. 그래서
$$m(t)=E\!\left[\frac{M(t)-(M(t)\bmod 2)}{2}\right]=\frac t2-\frac12P(M(t)\text{ odd})=\frac t2-\frac14+\frac14e^{-2t}$$
가 정확히 나오고 ($P(M \text{ odd})=(1-e^{-2t})/2$), 상수항 $-1/4=(\sigma^2-\mu^2)/(2\mu^2)$은 '주기가 지수분포보다 규칙적이면 첫 주기를 손해 본다'는 뜻이다.

수명 교체에서는 $T$를 줄이면 고장(비싼 교체)은 줄지만 주기가 짧아져 교체 횟수가 늘어난다; IFR이면 두 효과가 정확히 한 점에서 균형을 이룬다.

### 2.7 함정(traps)

1. $N(t)/t\to1/\mu$ a.s.에서 $E[N(t)]/t\to1/\mu$가 '자동으로' 나오지 않는다. 기본 재생 정리는 별도의 정리이며 증명에 Wald 항등식(정지시간 $N(t)+1$)이 필요하다.
2. $E[S_{N(t)}]\ne\mu\,m(t)$. $N(t)$는 정지시간이 아니다. 옳은 식은 $E[S_{N(t)+1}]=\mu(m(t)+1)$. Erlang(2,1), $t=50$에서 두 값이 실제로 얼마나 다른지는 E2에서 시뮬레이션으로 확인한다.
3. 재생-보상의 극한은 $E[R]/E[X]$이지 $E[R/X]$가 아니다. 비용률 곡선을 '주기별 (비용/길이)'의 평균으로 계산하면 틀린다(Fig 4의 점선).
4. $m(t)\approx t/\mu+(\sigma^2-\mu^2)/(2\mu^2)$: 상수항은 $\sigma<\mu$이면 음수다. 시뮬레이션에서 $m(t)-t/\mu$가 0이 아니라 $-1/4$로 가는 것을 보고 버그로 오해하지 말 것.
5. `renewal_reward`는 지평 $T$까지 '완료된' 주기의 보상만 합친다. 마지막 미완 주기 때문에 $O(1/T)$ 편향이 있으므로 지평을 충분히 크게 잡아야(여기서는 2000, 편향 $\approx-0.0005$) 4 SE 비교가 의미 있다.
6. 수명 교체는 $c_1<c_2$ 그리고 IFR일 때만 유한한 $T^*$가 있다. 지수 수명이면 무기억성 때문에 '새 부품'이 '쓰던 부품'보다 나을 게 없어 예방 교체는 순손해다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

- '재생-보상 정리가 왜 $E[R]/E[X]$이지 $E[R/X]$가 아닌지'를 주기 3개짜리 구체적 숫자 예로 스스로 설명해 보세요.
- 수명 교체에서 $T\to0$일 때 $C(T)\to\infty$인 이유를 한 문장으로 적어 보세요.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향(위/아래)이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # Erlang(2,1) 도착 간격(평균 mu=2, 분산 2)으로 만든 재생 과정에서 N(t)/t 의 극한은?
    "N_over_t_limit": None,
    # 같은 과정에서 E[N(50)] 은? (힌트: t/mu 에서 얼마나 벗어날까? 위? 아래?)
    "E_N_50": None,
    # P(N(50) >= 30) 은? (힌트: 대략 정규분포, 분산은 sigma^2 t / mu^3)
    "P_N50_ge_30": None,
    # 수명 X ~ Weibull(모양 2, 척도 1) (Fbar(x)=exp(-x^2), mu≈0.886), c1=1, c2=5 일 때
    # 최적 예방 교체 나이 T* 는? (mu 보다 클까 작을까?)
    "T_star": None,
    # 그때 최소 비용률 C(T*) 는?
    "C_star": None,
    # 예방 교체를 전혀 하지 않을 때 (T = inf) 의 비용률 c2/mu 는?
    "C_never": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 한 경로에서 $N(t)/t$의 수렴 (정리 1)

Erlang(2,1) = Gamma(shape 2, scale 1) 간격으로 재생 시각을 만들고 로그 $t$축에서 $N(t)/t$를 본다. 정리 1의 샌드위치 부등식 $S_{N(t)}\le t<S_{N(t)+1}$을 $t=10$에서 숫자로 확인한다.

In [ ]:
MU, SIG2 = 2.0, 2.0                       # Erlang(2,1): 평균 2, 분산 2
sampler = lambda n, g: g.gamma(2.0, 1.0, n)
T_long = scale(2000)                      # FAST: 200
times = renewal_times(sampler, T_long, rng)
t_grid = np.logspace(0, np.log10(T_long), 200)
N_grid = renewal_count(times, t_grid)

fig, ax = plt.subplots()
ax.plot(t_grid, N_grid / t_grid, label="N(t)/t, one path")
ax.axhline(1 / MU, ls="--", color="C3", label=f"1/mu = {1/MU:g}")
ax.set_xscale("log"); ax.set_xlabel("t (log)"); ax.set_ylabel("N(t)/t")
ax.set_title("Fig 1: N(t)/t along one Erlang(2,1) renewal path")
ax.legend(); plt.show()

In [ ]:
t0 = 10.0
n0 = int(renewal_count(times, t0))
print(f"t = {t0}:  N(t) = {n0},  S_N(t) = {times[n0-1]:.3f} <= {t0} < S_N(t)+1 = {times[n0]:.3f}")
print(f"           S_N/N = {times[n0-1]/n0:.3f} <= t/N = {t0/n0:.3f} < S_(N+1)/N = {times[n0]/n0:.3f}   (all -> mu = 2)")

N_T = int(renewal_count(times, T_long))
se_clt = np.sqrt(SIG2 / (MU**3 * T_long))   # 재생 CLT: Var(N(t)) ~ sigma^2 t / mu^3
print(f"N(T)/T = {N_T/T_long:.4f}  (theoretical SE from the renewal CLT: {se_clt:.4f})")

### 4.2 다경로 $N(50)$의 정확한 분포 (Ross 12e §7.2)

`renewal_times`는 경로 하나만 만든다. 다경로 벡터화를 위해 간격 행렬의 누적합 $S$ (모양 `(n_paths, n_cols)`)를 직접 만든다. 정확한 pmf는 푸아송 패리티(parity) 구성 $N(t)=\lfloor M(t)/2\rfloor$에서 나온다: $P(N=k)=P(M=2k)+P(M=2k+1)$.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def renewal_counts_matrix(sampler_shape, n_paths, n_cols, rng):
    """i.i.d. 간격을 (n_paths, n_cols)로 뽑아 누적합 S 를 돌려준다. N(t) = (S <= t).sum(1)."""
    return sampler_shape((n_paths, n_cols), rng).cumsum(axis=1)

def erlang2_N_pmf(t, k_max):
    """Erlang(2,1) 재생 과정의 N(t) pmf: M ~ Poisson(t), N = floor(M/2)."""
    pm = stats.poisson.pmf(np.arange(2 * k_max + 2), t)
    k = np.arange(k_max + 1)
    return k, pm[2 * k] + pm[2 * k + 1]

In [ ]:
t_obs, n_cols = 50.0, 70          # n_cols=70: S[:, -1] ~ Gamma(140) 이므로 60 을 넘는 것이 사실상 확실
S = renewal_counts_matrix(lambda shape, g: g.gamma(2.0, 1.0, shape), N_PATHS, n_cols, rng)
assert (S[:, -1] > 60).all(), "n_cols 가 너무 작다"
N50 = (S <= t_obs).sum(1)

k, pmf = erlang2_N_pmf(t_obs, k_max=60)
mean_asym, var_asym = t_obs / MU, SIG2 * t_obs / MU**3      # 정규근사 N(25, 12.5)
fig, ax = plt.subplots()
plot_hist_vs_pmf(N50, k, pmf, ax=ax, label_pmf="exact pmf (Poisson parity)")
xs = np.linspace(5, 45, 300)
ax.plot(xs, stats.norm.pdf(xs, mean_asym, np.sqrt(var_asym)), "k--", label=f"normal approx N({mean_asym:g}, {var_asym:g})")
ax.set_xlim(5, 45); ax.set_xlabel("N(50)"); ax.set_ylabel("density")
ax.set_title(f"Fig 2: N(50) over {N_PATHS} Erlang(2,1) renewal paths")
ax.legend(); plt.show()

In [ ]:
est_N50 = mc_estimate(N50)
est_P30 = mc_proportion(N50 >= 30)
mean_exact = float((k * pmf).sum())
var_exact = float((k**2 * pmf).sum() - mean_exact**2)
P30_normal = stats.norm.sf((29.5 - mean_exact) / np.sqrt(var_exact))     # 정규근사 + 연속성 보정
print(f"E[N(50)]      MC {est_N50}")
print(f"P(N(50)>=30)  MC {est_P30}   normal approx (continuity-corrected) {P30_normal:.4f}")
print(f"Var(N(50))    sample {N50.var(ddof=1):.3f}   exact {var_exact:.4f}   asymptotic sigma^2 t/mu^3 = {var_asym:.2f}")

### 4.3 기본 재생 정리와 상수 편차 (정리 2, 함정 4)

같은 $S$ 행렬로 $t=1,\dots,60$에서 $\hat m(t)$를 구한다. 왼쪽: $\hat m(t)-t/\mu$가 0이 아니라 $-1/4$로 간다 (닫힌 형식 $-\frac14+\frac14e^{-2t}$). 오른쪽: $\hat m(t)/t\to1/\mu$ (기본 재생 정리).

In [ ]:
t_arr = np.arange(1, 61)
counts = np.stack([(S <= t).sum(1) for t in t_arr], axis=1)      # (n_paths, 60)
m_hat = counts.mean(0)
m_se = counts.std(0, ddof=1) / np.sqrt(N_PATHS)
offset = (SIG2 - MU**2) / (2 * MU**2)                            # = -1/4
m_erlang2 = lambda t: t / MU + offset + np.exp(-2 * t) / 4       # Erlang(2,1) 닫힌 형식 m(t) (2.6절)
m_exact = m_erlang2(t_arr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.fill_between(t_arr, m_hat - t_arr/MU - 2*m_se, m_hat - t_arr/MU + 2*m_se, alpha=0.3, label="MC +-2 SE")
ax1.plot(t_arr, m_hat - t_arr/MU, "C0.-", ms=4, label="MC m(t) - t/mu")
ax1.plot(t_arr, m_exact - t_arr/MU, "C1-", label="exact -1/4 + exp(-2t)/4")
ax1.axhline(offset, ls="--", color="C3", label="(sigma^2 - mu^2)/(2 mu^2) = -1/4")
ax1.set_xlabel("t"); ax1.set_ylabel("m(t) - t/mu"); ax1.set_title("Fig 3a: constant offset"); ax1.legend()
ax2.plot(t_arr, m_hat / t_arr, "C0.-", ms=4, label="MC m(t)/t")
ax2.plot(t_arr, m_exact / t_arr, "C1-", label="exact m(t)/t")
ax2.axhline(1/MU, ls="--", color="C3", label="1/mu")
ax2.set_xlabel("t"); ax2.set_ylabel("m(t)/t"); ax2.set_title("Fig 3b: elementary renewal theorem"); ax2.legend()
plt.tight_layout(); plt.show()

### 4.4 수명 교체 비용률 곡선 (정리 4)

공유 표본(common random numbers) 하나로 모든 $T$에서 비용률을 추정하면 곡선이 매끈해진다. 올바른 추정량은 (표본평균 비용)/(표본평균 길이); 함정 3의 틀린 추정량 mean(비용/길이)도 점선으로 같이 그린다. 정확한 $T^*$는 $C(T)$의 직접 최소화와 최적 조건식 $g(T)=c_1/(c_2-c_1)$의 근, 두 가지로 구해 일치를 확인한다.

In [ ]:
c1, c2 = 1.0, 5.0
MU_W = special.gamma(1.5)                                       # Weibull(2,1) 평균 = sqrt(pi)/2
Fbar_w = lambda x: np.exp(-x**2)
int_Fbar_w = lambda T: np.sqrt(np.pi) / 2 * special.erf(T)      # E[min(X,T)] = int_0^T Fbar

def cost_rate_exact(T, c1=c1, c2=c2):
    return (c1 * Fbar_w(T) + c2 * (1 - Fbar_w(T))) / int_Fbar_w(T)

def optimality_gap(T, c1=c1, c2=c2):                            # g(T) - c1/(c2-c1),  r(T) = 2T
    return 2 * T * int_Fbar_w(T) - (1 - Fbar_w(T)) - c1 / (c2 - c1)

T_star = optimize.minimize_scalar(cost_rate_exact, bounds=(0.05, 5), method="bounded",
                                  options={"xatol": 1e-10}).x
T_root = optimize.brentq(optimality_gap, 0.01, 5, xtol=1e-12)
np.testing.assert_allclose(T_star, T_root, atol=1e-6)           # 정확한 항등식: 최소점 == 조건식의 근
C_star, C_never = cost_rate_exact(T_star), c2 / MU_W
print(f"T* = {T_star:.6f} (root {T_root:.6f}),  C(T*) = {C_star:.6f} = (c2-c1) r(T*) = {(c2-c1)*2*T_star:.6f}")
print(f"C(inf) = c2/mu = {C_never:.6f},  saving {1 - C_star/C_never:.1%},  F(T*) = {1 - Fbar_w(T_star):.6f}")

In [ ]:
n_shared = scale(200_000)
X_shared = rng.weibull(2.0, n_shared)
T_grid = np.linspace(0.1, 3.0, 30)
C_hat = np.array([np.where(X_shared <= T, c2, c1).mean() / np.minimum(X_shared, T).mean() for T in T_grid])
C_wrong = np.array([(np.where(X_shared <= T, c2, c1) / np.minimum(X_shared, T)).mean() for T in T_grid])
frac_fail = mc_proportion(X_shared <= T_star)

fig, ax = plt.subplots()
T_fine = np.linspace(0.05, 3.0, 300)
ax.plot(T_fine, cost_rate_exact(T_fine), "C1-", label="exact C(T)")
ax.plot(T_grid, C_hat, "C0o", ms=4, label="MC ratio of means (shared sample)")
ax.plot(T_grid, C_wrong, "C2:", label="wrong: mean(cost/length)")
ax.axvline(T_star, color="C3", ls="-.", label=f"T* = {T_star:.3f}")
ax.axhline(C_never, color="k", ls="--", label=f"c2/mu = {C_never:.2f}")
ax.set_ylim(3, 12); ax.set_xlabel("replacement age T"); ax.set_ylabel("long-run cost rate")
ax.set_title("Fig 4: Age-replacement cost rate, Weibull(2,1), c1 = 1, c2 = 5")
ax.legend(); plt.show()
print(f"fraction of failure replacements at T*: {frac_fail}   (exact F(T*) = {1 - Fbar_w(T_star):.6f})")

### 4.5 `renewal_reward`로 정리 3 직접 검증

지평 $H$까지 완료된 주기의 비용을 합쳐 $R(H)/H$를 만들고, 독립 반복 $K$번의 평균으로 $C(T^*)$와 $C(\infty)=c_2/\mu$를 추정한다. 절단된 주기는 정확히 $T^*$이므로 `cycles < T_star`가 고장 교체다(동률 확률 0).

In [ ]:
K, H = scale(100, divisor=5, minimum=20), 2000
sampler_T = lambda n, g: np.minimum(g.weibull(2.0, n), T_star)
reward_fn = lambda cycles, g: np.where(cycles < T_star, c2, c1)
sampler_inf = lambda n, g: g.weibull(2.0, n)
reward_inf = lambda cycles, g: np.full(len(cycles), c2)

rates_Tstar = np.array([renewal_reward(sampler_T, reward_fn, H, rng) / H for _ in range(K)])
rates_inf = np.array([renewal_reward(sampler_inf, reward_inf, H, rng) / H for _ in range(K)])
est_Cstar, est_Cnever = mc_estimate(rates_Tstar), mc_estimate(rates_inf)
print(f"C(T*)  MC {est_Cstar}   exact {C_star:.6f}")
print(f"C(inf) MC {est_Cnever}   exact {C_never:.6f}")

In [ ]:
T_path = 3000
fig, ax = plt.subplots()
for smp, lbl, ref, col in [(sampler_T, "age replacement at T*", C_star, "C0"),
                           (sampler_inf, "never replace preventively", C_never, "C1")]:
    tp = renewal_times(smp, T_path, rng)
    gaps = np.diff(np.concatenate(([0.0], tp)))          # cumsum -> diff 는 ~1e-13 의 반올림 오차를 남긴다
    is_fail = gaps < T_star - 1e-9                       # 그래서 '정확히 T*' 인 예방 교체를 허용오차로 판별
    costs = np.where(is_fail, c2, c1) if smp is sampler_T else np.full(len(gaps), c2)
    ax.plot(tp, np.cumsum(costs) / tp, color=col, lw=1, label=f"cumulative cost / t: {lbl}")
    ax.axhline(ref, color=col, ls="--", label=f"limit {ref:.3f}")
ax.set_xscale("log"); ax.set_ylim(3, 8)
ax.set_xlabel("t (log)"); ax.set_ylabel("R(t)/t")
ax.set_title("Fig 5: Renewal-reward convergence along one path")
ax.legend(); plt.show()

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
m50_exact = m_erlang2(t_obs)                                      # Erlang(2,1) 닫힌 형식 (4.3절)
P30_exact = float(pmf[k >= 30].sum())                             # 푸아송 패리티 pmf
F_Tstar = 1 - Fbar_w(T_star)
pred_T = predictions["T_star"]
print(f"T* (MC 항목 아님):  내 예측 {'—' if pred_T is None else f'{pred_T:.4f}'}   정확값 {T_star:.6f}")
rows = [
    {"name": f"N(T)/T, single path (T = {T_long})", "predicted": predictions["N_over_t_limit"],
     "estimate": N_T / T_long, "se": se_clt, "exact": 1 / MU},
    {"name": "E[N(50)]", "predicted": predictions["E_N_50"], "estimate": est_N50, "exact": m50_exact},
    {"name": "P(N(50) >= 30)", "predicted": predictions["P_N50_ge_30"], "estimate": est_P30, "exact": P30_exact},
    {"name": "C(T*) age-replacement cost rate", "predicted": predictions["C_star"], "estimate": est_Cstar, "exact": C_star},
    {"name": "C(inf) = c2/mu, never replace", "predicted": predictions["C_never"], "estimate": est_Cnever, "exact": C_never},
    {"name": "fraction of failure replacements at T*", "predicted": None, "estimate": frac_fail, "exact": F_Tstar},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **N(T)/T 단일 경로**: 어긋나면 `renewal_times`의 간격 생성 또는 재생 CLT의 분산 $\sigma^2t/\mu^3$을 잘못 적용한 것이다.
- **E[N(50)]**: 어긋나면 푸아송 패리티 유도(또는 $S$ 행렬의 누적합 방향)가 틀린 것이다. 24.75가 25가 아님에 주목(함정 4).
- **P(N(50) ≥ 30)**: 어긋나면 패리티 pmf의 인덱스($2k$, $2k+1$)나 부등호 `>=`를 의심한다. 4.2절에서 출력한 연속성 보정 정규근사는 정확값과 약 0.002 차이 — MC의 SE와 같은 크기이므로 이 표는 둘을 구별하지 못한다. 정확 pmf를 쓰는 이유는 '있어서'이지 표가 요구해서가 아니다.
- **T\***: 위에 따로 출력했다. MC 항목이 아니라 정확 항등식(최소점 = 조건식의 근)으로 이미 검증했으므로 표에는 넣지 않는다.
- **C(T\*)**: 어긋나면 `renewal_reward`의 완료 주기 편향($O(1/H)$) 또는 `reward_fn`의 부등호 방향 문제다.
- **C(∞)**: 어긋나면 Weibull 평균 $\Gamma(3/2)$ 계산이나 `rng.weibull`의 매개변수화(척도 1, 모양 2)를 의심한다.
- **고장 교체 비율**: 어긋나면 $\bar F(x)=e^{-x^2}$ 형태(즉 `rng.weibull(2.0)`의 정의)가 틀린 것이다.

In [ ]:
assert_close(N_T / T_long, 1 / MU, se=se_clt, k=4, name="N(t)/t single path")
assert_close(est_N50, m50_exact, k=4, name="E[N(50)] Erlang(2,1)")
assert_close(est_P30, P30_exact, k=4, name="P(N(50)>=30)")
assert_close(est_Cstar, C_star, k=4, name="C(T*) via renewal_reward")
assert_close(est_Cnever, C_never, k=4, name="C(inf) = c2/mu")
assert_close(frac_fail, F_Tstar, k=4, name="F(T*) failure fraction")
np.testing.assert_allclose(T_star, T_root, atol=1e-6)   # exact identity (not MC)
print("all checks passed")

## 6. 연습문제

### E1 계산

수명 $X\sim$ Uniform(0,1), $c_1=1$, $c_2=5$.
(a) $C(T)$를 닫힌 형식으로 써라.
(b) 최적 조건식 $g(T)=c_1/(c_2-c_1)$으로 $T^*$와 $C(T^*)$를 손으로 구하라.
(c) 예방 교체를 하지 않을 때($c_2/\mu$)와 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\bar F(x)=1-x$, $\int_0^T\bar F=T-T^2/2$, $F(T)=T$이므로
$$C(T)=\frac{1\cdot(1-T)+5T}{T-T^2/2}=\frac{1+4T}{T-T^2/2},\qquad 0<T\le1.$$

(b) $r(T)=\dfrac{1}{1-T}$ (균등분포는 IFR). $g(T)=\dfrac{T-T^2/2}{1-T}-T=\dfrac{T-T^2/2-T+T^2}{1-T}=\dfrac{T^2/2}{1-T}$. 조건 $g(T)=\dfrac{c_1}{c_2-c_1}=\dfrac14$ ⟺ $2T^2=1-T$ ⟺ $2T^2+T-1=0$ ⟺ $T^*=\dfrac{-1+3}{4}=\dfrac12$. 최소값 $C(T^*)=(c_2-c_1)r(T^*)=4\cdot2=8$ (직접: $C(1/2)=3/0.375=8$).

(c) $\mu=1/2$이므로 $c_2/\mu=10$. $T^*=1/2$로 예방 교체하면 20% 절감.

검산 코드:
```python
C_unif = lambda T: (1 + 4 * T) / (T - T**2 / 2)
res = optimize.minimize_scalar(C_unif, bounds=(0.01, 0.99), method="bounded")
print(res.x, res.fun, C_unif(0.5))     # 0.5000  8.0000  8.0
```

</details>

### E2 유도 후 시뮬레이션 검증

Erlang(2,1) 재생 과정, $t=50$.
(a) $N(t)+1$이 정지시간임을 보이고 Wald 항등식으로 $E[S_{N(t)+1}]$을 계산하라.
(b) $E[S_{N(t)}]$는 $\mu\,m(t)$와 같은가? 4.2절의 $S$ 행렬을 재사용해 두 값을 추정하고 비교하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\{N(t)+1=n\}=\{S_{n-1}\le t<S_n\}\in\sigma(X_1,\dots,X_n)$이므로 정지시간이다. Wald: $E[S_{N(t)+1}]=\mu(m(t)+1)=2\times25.75=51.5$.

(b) $N(t)$는 정지시간이 아니므로 ($\{N(t)=n\}=\{S_n\le t<S_{n+1}\}$은 $X_{n+1}$까지 봐야 한다) Wald가 적용되지 않는다. $E[S_{N(t)}]=t-E[A(t)]$ ($A(t)$ = 나이)이고 $E[A(t)]\to E[X^2]/(2\mu)=(2+4)/4=1.5$ (04b에서 유도; $t=50$이면 오차는 $e^{-50}$ 수준)이므로 $E[S_{N(t)}]\approx48.5\ne\mu\,m(t)=49.5$.

```python
idx = (S <= 50).sum(1)                       # N(50) per path
rows_ = np.arange(N_PATHS)
S_N1 = S[rows_, idx]                         # S_{N(t)+1}
S_N = np.where(idx > 0, S[rows_, np.maximum(idx - 1, 0)], 0.0)   # S_{N(t)} (S_0 = 0)
print(mc_estimate(S_N1), "  Wald:", MU * (m50_exact + 1))       # 51.50 ± 0.01 (full)
print(mc_estimate(S_N), "  mu*m(t) =", MU * m50_exact, " t - 1.5 =", 50 - 1.5)   # 48.50 ± 0.01
```

</details>

### E3 가정을 깨보기

수명을 지수분포(비율 1)로 바꾸고 $c_1=1,c_2=5$에서 4.4절의 비용률 곡선을 다시 그려라. $T^*$는 존재하는가? 최적 조건식의 좌변 $g(T)$를 계산해 무엇이 일어나는지 설명하라. (보너스) Weibull 모양 0.5(DFR)에서는?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

지수분포(비율 $r=1$): $\bar F(T)=e^{-T}$, $\int_0^T\bar F=1-e^{-T}$,
$$C(T)=\frac{e^{-T}+5(1-e^{-T})}{1-e^{-T}}=5+\frac{e^{-T}}{1-e^{-T}},$$
$T$에 대해 단조감소하고 $T\to\infty$에서 $c_2/\mu=5$. 최적 조건식의 좌변은
$$g(T)=r\int_0^Te^{-rx}dx-F(T)=(1-e^{-rT})-(1-e^{-rT})\equiv0<\tfrac14,$$
해가 없다 → $T^*=\infty$ (예방 교체 없음이 최적). 이유는 무기억성: 나이 든 부품의 잔여수명이 새 부품과 같은 분포이므로 예방 교체는 비용만 든다. 숫자: $C(0.5)=6.541$, $C(1)=5.582$, $C(3)=5.052$.

보너스(DFR, Weibull 모양 0.5): 오래된 부품이 오히려 더 좋으므로($r$ 감소) $C$는 더 급하게 감소하고 역시 $T^*=\infty$.

```python
X_exp = rng.exponential(1.0, n_shared)
C_hat_exp = np.array([np.where(X_exp <= T, c2, c1).mean() / np.minimum(X_exp, T).mean() for T in T_grid])
C_exp = lambda T: c2 + np.exp(-T) / (1 - np.exp(-T))
g_exp = lambda T: 1.0 * (1 - np.exp(-T)) - (1 - np.exp(-T))       # identically 0
print(C_exp(np.array([0.5, 1.0, 3.0])), np.abs(g_exp(T_grid)).max())   # [6.541 5.582 5.052]  0.0
fig, ax = plt.subplots()
ax.plot(T_grid, C_exp(T_grid), "C1-", label="exact C(T), exponential")
ax.plot(T_grid, C_hat_exp, "C0o", ms=4, label="MC ratio of means")
ax.axhline(c2 / 1.0, color="k", ls="--", label="c2/mu = 5"); ax.set_ylim(4.5, 8)
ax.set_xlabel("T"); ax.set_ylabel("cost rate"); ax.legend(); plt.show()
```

</details>

## 7. 정리

1. 재생 과정은 i.i.d. 주기의 열이고, 장기 비율 문제는 모두 '한 주기의 기댓값 비'로 환원된다: $N(t)/t\to1/\mu$, $R(t)/t\to E[R]/E[X]$.
2. a.s. 수렴(강대수법칙)과 기댓값 수렴(기본 재생 정리)은 별개의 정리이며, 후자는 정지시간 $N(t)+1$과 Wald 항등식 $E[S_{N(t)+1}]=\mu(m(t)+1)$로 증명한다.
3. Erlang(2,1)에서는 $N(t)=\lfloor M(t)/2\rfloor$이므로 $m(t)=t/2-1/4+e^{-2t}/4$: 상수 편차 $(\sigma^2-\mu^2)/(2\mu^2)=-1/4$가 눈에 보인다.
4. 수명 교체: $C(T)=\frac{c_1\bar F(T)+c_2F(T)}{\int_0^T\bar F}$, 위험률이 순증가(강한 IFR)하고 해가 존재하면 $r(T^*)\int_0^{T^*}\bar F-F(T^*)=\frac{c_1}{c_2-c_1}$의 유일한 해가 최적이고 $C(T^*)=(c_2-c_1)r(T^*)$; Weibull(2,1), $c_1=1,c_2=5$: $T^*=0.511$, $C^*=4.085<5.642$.
5. 지수 수명이면 무기억성 때문에 예방 교체는 항상 손해($T^*=\infty$) — 정리의 가정(IFR)이 왜 필요한지 보여 주는 반례.

**trap 카드**
- Q: 재생-보상 정리로 비용률을 추정할 때 각 주기의 (비용/길이)를 평균 내면 왜 틀리는가?
- A: 정리의 극한은 $E[R]/E[X]$(기댓값의 비)이지 $E[R/X]$(비의 기댓값)가 아니다. 긴 주기는 시간축에서 더 큰 가중치를 받으므로 주기별 비를 단순 평균하면 짧은 주기가 과대평가된다(Fig 4의 점선). 올바른 추정량은 (총 비용)/(총 시간) 또는 (표본평균 비용)/(표본평균 길이).

**다음 노트북: 04b** — 같은 재생-보상 논법으로 '시각 $t$에 관찰한 나이·잔여수명·구간 길이'의 극한분포(평형분포 $F_e$)를 유도하고, 02d에서 맛본 검사 역설을 일반 분포로 확장한다. E2에서 만난 $E[S_{N(t)}]\approx t-1.5$의 1.5가 바로 그 답이다.

log/progress.md 한 줄 템플릿:
`- YYYY-MM-DD 04a 기본 재생 정리와 재생-보상 | 소요 __분 | 예측 적중 _/6 | 막힌 곳: ___ | 다음: 04b`